# Herding, Momentum, and Reversal Strategy Implementation

This notebook implements a quantitative trading strategy inspired by the paper "Herding, Momentum, and Reversal in China's A-Share Market: An Agent-Based Network Model with Information Diffusion" by Jiahao Weng (2026). The strategy aims to capture momentum and reversal effects driven by herding behavior and information diffusion delays.

**Paper Citation:**
Weng, J. (2026). Herding, Momentum, and Reversal in China's A-Share Market: An Agent-Based Network Model with Information Diffusion. *arXiv preprint arXiv:2607.27063*.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration

# Trading universe (S&P 500 subset)
UNIVERSE = ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'TSLA', 'FB', 'BRK-B', 'JPM', 'JNJ', 'V']

# Strategy parameters
LOOKBACK_PERIOD = 20
MOMENTUM_THRESHOLD = 1.5
REVERSAL_THRESHOLD = -1.0
POSITION_SIZE = 0.02  # 2% of portfolio per position

# Hypothesis
"""
The strategy aims to profit from momentum and reversal effects driven by herding behavior and information diffusion delays.
Momentum is identified when recent returns are significantly positive, while reversal is identified when recent returns are significantly negative.
"""


## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download historical data
data = yf.download(UNIVERSE, start='2010-01-01', end='2023-01-01', group_by='ticker')

# Compute momentum and reversal signals
def compute_signals(data, lookback):
    signals = pd.DataFrame(index=data['Close'].index)
    for ticker in UNIVERSE:
        returns = data['Close'][ticker].pct_change().dropna()
        momentum = returns.rolling(lookback).mean()
        signals[ticker] = np.where(momentum > MOMENTUM_THRESHOLD, 1,
                                   np.where(momentum < REVERSAL_THRESHOLD, -1, 0))
    return signals

signals = compute_signals(data, LOOKBACK_PERIOD)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Generate portfolio weights
def generate_weights(signals):
    weights = signals.shift(1).dropna() * POSITION_SIZE
    weights = weights.fillna(0)
    weights = weights.div(weights.sum(axis=1), axis=0)  # Normalize to sum to 1
    return weights

weights = generate_weights(signals)

## Phase 4 — Vectorized Backtest

In [ ]:
# Compute portfolio returns
portfolio_returns = (weights * data['Close'].pct_change()).sum(axis=1)

# Compute cumulative returns
cumulative_returns = (1 + portfolio_returns).cumprod()


## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Performance metrics
annual_return = portfolio_returns.mean() * 252
annual_volatility = portfolio_returns.std() * np.sqrt(252)
sharpe_ratio = annual_return / annual_volatility
sortino_ratio = annual_return / portfolio_returns[portfolio_returns < 0].std() * np.sqrt(252)
max_drawdown = (cumulative_returns / cumulative_returns.cummax() - 1).min()
calmar_ratio = annual_return / abs(max_drawdown)

# Print metrics
print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')

# Plot equity curve
plt.plot(cumulative_returns)
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.show()

## Phase 6 — Monitoring Stub

In [ ]:
# Monitoring function
def monitor_portfolio(data, weights):
    current_prices = data['Close'].iloc[-1]
    current_positions = weights.iloc[-1]
    portfolio_value = (current_prices * current_positions).sum()
    pnl = portfolio_value - 1  # Assuming initial portfolio value of 1
    print(f'Daily P&L: {pnl:.2%}')
    print('Current Positions:')
    print(current_positions[current_positions!= 0])

# Example usage
monitor_portfolio(data, weights)